# Phase 3 — Train: Cascade R-CNN + ConvNeXt-Tiny

Notebook này **mỏng**: mọi logic nằm trong `scripts/` và `src/`. Sửa logic thì
sửa file `.py` rồi `git pull`, không sửa trong notebook.

## Chạy theo thứ tự này

| Ô | Việc | Mất bao lâu |
|---|---|---|
| [3.1] | Khai báo đường dẫn | — |
| [3.2] | Mount Drive | vài giây |
| [3.3] | Lấy code từ GitHub | vài giây |
| [3.4] | Kiểm môi trường mmdet | vài giây |
| [3.5] | Giải nén dataset vào `/content` | 1–2 phút |
| [3.6] | **Kiểm tra dữ liệu** | ~1 phút |
| [3.7] | Đo dải anchor của RPN | vài giây |
| [3.8] | Sinh 20 ảnh overfit | vài giây |
| [3.9] | **Đo thời gian 1 epoch + VRAM** (GATE 3) | ~5 phút |
| [3.10] | Chạy sanity check 20 ảnh (GATE 3) | 20–40 phút |
| [3.11] | **Train thật** — chỉ sau khi GATE 3 đạt | 6–8 giờ |

## ⚠️ Bắt buộc: chạy notebook 00 ô [0.6] trước

Notebook này **không cài** mmcv/mmengine/mmdet/mmpretrain. Môi trường do
notebook 00 dựng, kèm ba miếng vá mà không có thì `import mmdet` nổ ngay
(xem `docs/NOTES.md` mục 1.2). `/content` mất sạch mỗi lần Colab reset runtime,
nên **mỗi phiên mới đều phải chạy lại ô [0.6] của notebook 00**.

Ô [3.4] sẽ kiểm và báo rõ nếu thiếu, chứ không để bạn đoán.

## Điều notebook này KHÔNG làm

Không chạy test trên tập test. Phase 5 mới là lúc đó, và **chỉ chạy đúng một
lần**. Tập test ở đây chỉ được kiểm tra về mặt cấu trúc (đếm ảnh, đếm box),
không được dùng để chọn tham số.


In [ ]:
# [3.1] CẤU HÌNH — sửa ở đây, không sửa trong code
REPO_URL = "https://github.com/21xDHoang/flood-house-counting.git"
REPO_DIR = "/content/flood-house-counting"

DRIVE_DIR     = "/content/drive/MyDrive/Flood_House_AI"
PROCESSED_ZIP = f"{DRIVE_DIR}/processed/floodnet_coco.zip"   # Phase 2 đóng gói
PROCESSED_DIR = "/content/floodnet_coco"                     # giải nén vào đây

# Đường dẫn TƯƠNG ĐỐI trong repo (script tự tìm gốc repo theo vị trí file).
CFG_CHINH   = "configs/mmdet/cascade_convnext_t_floodnet.py"
CFG_OVERFIT = "configs/mmdet/overfit20.py"
DATA_YAML   = "configs/data.yaml"

# Số epoch cho lần train thật. ĐANG ĐỂ MẶC ĐỊNH 24 — con số này CHỐT Ở GATE 3,
# sau khi ô [3.9] cho biết 1 epoch mất bao lâu và VRAM còn dư bao nhiêu.
# `scripts/train.py` sẽ tự chỉnh mốc giảm LR theo cùng tỉ lệ nếu đổi số ở đây.
SO_EPOCH = 24

# Số vòng lặp chạy thử ở ô [3.9]. 20 vòng là đủ để bỏ 2 vòng khởi động và lấy
# trung bình 18 vòng.
SO_VONG_DO = 20

print("Đã nạp cấu hình Phase 3.")
print(f"  repo      : {REPO_URL}")
print(f"  dataset   : {PROCESSED_DIR}  <- giải nén từ {PROCESSED_ZIP}")
print(f"  config    : {CFG_CHINH}")
print(f"  số epoch  : {SO_EPOCH}  (chốt ở GATE 3)")


In [ ]:
# [3.2] GẮN GOOGLE DRIVE
#
# Checkpoint và dataset đều nằm trên Drive. `/content` chỉ là đĩa tạm và mất
# sạch khi hết phiên, nên checkpoint BẮT BUỘC phải ghi ra Drive.
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# [3.3] LẤY CODE TỪ GITHUB
#
# /content mất sạch mỗi lần runtime reset, nên phiên nào cũng phải làm bước này.
# Có sẵn repo rồi thì `git pull` cho nhanh.
import os
import subprocess


def chay(lenh, **them):
    """Chạy lệnh, in stdout; trả về mã thoát."""
    kq = subprocess.run(lenh, capture_output=True, text=True, **them)
    if kq.stdout.strip():
        print(kq.stdout.strip())
    if kq.returncode != 0:
        print("LỖI:", kq.stderr.strip())
    return kq.returncode


if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Đã có repo, cập nhật code mới nhất:")
    chay(["git", "-C", REPO_DIR, "pull"])
else:
    print("Chưa có repo, clone về:")
    chay(["git", "clone", REPO_URL, REPO_DIR])

print()
print("Commit đang dùng:")
chay(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])

# Kiểm repo đã có đủ code Phase 3 chưa. Clone từ lúc chưa push sẽ thiếu file, và
# khi đó script thoát ngay mà không in gì — rất khó đoán ra.
print()
print("Kiểm tra code Phase 3:")
CAN_CO = ["configs/mmdet/cascade_convnext_t_floodnet.py",
          "configs/mmdet/overfit20.py",
          "configs/data.yaml",
          "scripts/train.py",
          "scripts/kiem_tra_du_lieu.py",
          "scripts/kiem_anchor.py",
          "scripts/tao_overfit20.py",
          "src/floodcount/data/kiem_tra.py",
          "src/floodcount/data/overfit.py",
          "src/floodcount/models/transforms.py",
          "src/floodcount/models/anchor.py"]
thieu = [d for d in CAN_CO if not os.path.exists(os.path.join(REPO_DIR, d))]
for d in CAN_CO:
    print(f"  [{'OK  ' if d not in thieu else 'THIẾU'}] {d}")
if thieu:
    raise SystemExit(f"\nThiếu {len(thieu)} tệp trong repo — code chưa được push "
                     f"hết lên GitHub. Báo lại cho Claude.")


In [ ]:
# [3.4] KIỂM MÔI TRƯỜNG + hàm chạy lệnh
import importlib
import subprocess
import sys

# Notebook này KHÔNG cài gì cả — môi trường do notebook 00 ô [0.6] dựng. Kiểm ở
# đây để báo rõ thiếu gì, thay vì để lỗi hiện ra ở giữa lúc dựng model với thông
# báo trỏ sai chỗ.
CAN_IMPORT = ["torch", "mmcv", "mmengine", "mmdet", "mmpretrain", "pycocotools"]
thieu = []
for ten in CAN_IMPORT:
    try:
        m = importlib.import_module(ten)
        print(f"{ten:<12}: {getattr(m, '__version__', '?')}")
    except Exception as e:                      # noqa: BLE001
        # Bắt rộng: import mmdet còn nổ AssertionError khi mmcv chưa được vá,
        # chứ không chỉ ImportError.
        print(f"{ten:<12}: THIẾU/LỖI -> {type(e).__name__}: {e}")
        thieu.append(ten)

if thieu:
    raise SystemExit(
        "\n" + "=" * 72 +
        "\n[!] Môi trường chưa sẵn sàng: " + ", ".join(thieu) +
        "\n" +
        "=" * 72 +
        "\n=> Chạy notebook 00 (00_colab_setup.ipynb), ô [0.2] rồi ô [0.6]," +
        "\n   SAU ĐÓ quay lại chạy ô [3.2] của notebook này." +
        "\n   /content mất sạch mỗi lần Colab reset runtime nên mỗi phiên mới" +
        "\n   đều phải cài lại." +
        "\n   Nếu mmdet báo AssertionError về mmcv: đó là miếng vá ngưỡng mmcv" +
        "\n   trong ô [0.6] chưa được áp — xoá /content/.floodcount_env_ready" +
        "\n   rồi chạy lại ô [0.6]." +
        "\n   Nếu mmpretrain báo TypeError 'NoneType takes no arguments': miếng" +
        "\n   vá WITH_MULTIMODAL chưa được áp — cách xử lý y hệt.")


def chay_hien_dan(lenh, cwd):
    """Chạy lệnh con và in từng dòng NGAY khi nó nhả ra, trả về mã thoát.

    KHÔNG dùng thẳng `subprocess.run(lenh, cwd=...)` cho các bước chạy dài.
    Trên Colab, fd 1 của tiến trình con không chảy vào ô output: script chạy
    xong, mã thoát 0, mà ô output trống trơn, không biết nó làm được gì.

    Phải tự đọc ống dẫn rồi in qua stdout của kernel thì mới thấy. Cách này vẫn
    giữ được log hiện dần chứ không dồn tới lúc chạy xong.

    Đối số `-u` truyền cho python của tiến trình con là bắt buộc: không có nó
    thì chính tiến trình con lại đệm output theo khối, và ta ngồi nhìn màn hình
    im lặng suốt vài phút rồi output mới ập ra một lúc.
    """
    kq = subprocess.Popen(lenh, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for dong in kq.stdout:
        print(dong, end="")
    kq.wait()
    return kq.returncode


print("\nSẵn sàng.")


---
## Bước 1 — Dataset

Dataset Phase 2 nằm trên Drive dưới dạng `floodnet_coco.zip` (~1–2 GB). Giải nén
vào `/content` (đĩa cục bộ) chứ **không đọc thẳng từ Drive**: mỗi vòng lặp train
phải đọc ảnh, mà đọc qua FUSE của Drive chậm hơn nhiều lần so với đĩa cục bộ.


In [ ]:
# [3.5] GIẢI NÉN DATASET VÀO /content
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4]. "
                     "Hãy chạy lần lượt từ ô [3.1] trở xuống rồi thử lại.")

import os
import shutil
import zipfile

DUONG_TRAIN_JSON = f"{PROCESSED_DIR}/annotations/instances_train.json"


def dem_dataset(goc):
    """(số ảnh trong ba split, số box) — (0, 0) nếu không phải dataset."""
    import json
    so_anh = so_box = 0
    for split in ("train", "val", "test"):
        duong = f"{goc}/annotations/instances_{split}.json"
        if not os.path.exists(duong):
            return 0, 0
        with open(duong, encoding="utf-8") as f:
            c = json.load(f)
        so_anh += len(c["images"])
        so_box += len(c["annotations"])
    return so_anh, so_box


if os.path.exists(DUONG_TRAIN_JSON):
    print("Đã có dataset trong phiên này, bỏ qua bước giải nén.")
else:
    if not os.path.exists(PROCESSED_ZIP):
        raise SystemExit(
            f"[!] Không thấy {PROCESSED_ZIP}\n"
            f"    Phase 2 chưa đóng gói dataset, hoặc Drive chưa mount (ô [3.2]).\n"
            f"    Chạy notebook 02 ô [2.8] để nén, rồi quay lại đây.")

    dung_luong = os.path.getsize(PROCESSED_ZIP) / 1024 ** 3
    trong = shutil.disk_usage("/content").free / 1024 ** 3
    print(f"zip       : {PROCESSED_ZIP}  ({dung_luong:.2f} GB)")
    print(f"/content  : còn trống {trong:.1f} GB")
    if trong < dung_luong * 2.5:
        print("[!] Còn ít chỗ trống. Giải nén cần khoảng gấp đôi dung lượng zip "
              "(tệp nén sẵn, gần như không co lại được).")

    print("\nĐang giải nén...")
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    with zipfile.ZipFile(PROCESSED_ZIP) as z:
        z.extractall(PROCESSED_DIR)

so_anh, so_box = dem_dataset(PROCESSED_DIR)
print(f"\ndataset: {PROCESSED_DIR}")
print(f"  {so_anh:,} ảnh / {so_box:,} box")

# Con số kỳ vọng của Phase 2 (xem docs/NOTES.md): 2343 ảnh / 6301 box. Lệch
# nghĩa là zip khác bản đã kiểm ở GATE 2 — dừng lại trước khi train.
if (so_anh, so_box) != (2343, 6301):
    print(f"\n[!] Số liệu KHÁC kỳ vọng của GATE 2 (2343 ảnh / 6301 box).")
    print("    Có thể zip này là bản dựng cũ. Kiểm lại trước khi train — model")
    print("    học trên tập khác thì mọi con số so sánh sau này đều vô nghĩa.")

for split in ("train", "val", "test"):
    n = len(os.listdir(f"{PROCESSED_DIR}/images/{split}"))
    print(f"  ảnh {split:<5}: {n:,}")

print(f"\nVRAM/đĩa xong. Bước tiếp theo: ô [3.6] kiểm tra dữ liệu.")


In [ ]:
# [3.6] KIỂM TRA DỮ LIỆU — bước chặn cuối trước khi train
#
# Đây là ô quan trọng nhất của cả notebook. Nó đối chiếu TÊN LỚP giữa data.yaml,
# config train và file COCO, rồi DỰNG THẬT dataset bằng mmdet để đếm lại số box.
#
# Vì sao phải đếm lại: mmdet có ba kiểu hỏng KHÔNG báo lỗi — (a) tên lớp trong
# `metainfo` không có trong JSON -> bỏ im lặng mọi box của lớp đó; (b) thứ tự
# category lệch -> bảng kết quả in ngược tên ngập/không ngập; (c) box có toạ độ
# vô lý -> bỏ im lặng. Cả ba đều chỉ hiện ra dưới dạng "mAP thấp khó hiểu" sau
# 6-8 giờ train.
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import sys

lenh_kiem = [sys.executable, "-u", "scripts/kiem_tra_du_lieu.py",
             "--config", DATA_YAML,
             "--mmdet-config", CFG_CHINH,
             "--processed-dir", PROCESSED_DIR,
             "--splits", "train,val,test",
             "--mmdet", "co"]
print("Chạy: " + " ".join(lenh_kiem))
print("=" * 72)
ma = chay_hien_dan(lenh_kiem, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma, "(0 là ĐẠT HẾT)")

if ma != 0:
    print("\n" + "!" * 72)
    print("DỪNG LẠI. Dữ liệu có vấn đề thì train xong cũng vô nghĩa.")
    print("Copy nguyên phần [LỖI] ở trên gửi lại cho Claude.")
    print("!" * 72)


In [ ]:
# [3.7] ĐO DẢI ANCHOR CỦA RPN
#
# RPN sinh anchor theo công thức cố định trong config, không liên quan gì tới dữ
# liệu. Nếu dải anchor không phủ tới cỡ nhà của FloodNet (trung vị cạnh 186px,
# p90 348px — đo ở Phase 2) thì những căn nhà to nhất không bao giờ khớp được với
# anchor nào. Model vẫn chạy, loss vẫn giảm, chỉ có recall thấp một cách khó hiểu.
#
# Ô này đọc tham số anchor TỪ CHÍNH config đồ án (đã phân giải `_base_`), không
# chép tay — vì config đồ án thừa hưởng RPN từ config mmdet nên nhìn vào file .py
# của đồ án sẽ không thấy `scales`/`ratios`.
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import sys

lenh_anchor = [sys.executable, "-u", "scripts/kiem_anchor.py",
               "--config", CFG_CHINH,
               "--processed-dir", PROCESSED_DIR,
               "--split", "train"]
print("Chạy: " + " ".join(lenh_anchor))
print("=" * 72)
ma_anchor = chay_hien_dan(lenh_anchor, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_anchor, "(0 là chạy được)")


---
## Bước 2 — GATE 3: đường ống có học được không?

Ba việc, làm đúng thứ tự:

1. **Sinh 20 ảnh overfit** ([3.8]) — chọn tất định, cùng seed thì cùng 20 ảnh.
2. **Đo thời gian 1 epoch + VRAM** ([3.9]) — để chốt số epoch, và để biết cấu
   hình có vừa VRAM không. Ô này **không ghi checkpoint nào**.
3. **Chạy sanity check 20 ảnh** ([3.10]) — train rồi đánh giá trên chính 20 ảnh
   đó. Đây là phép thử "học vẹt": nếu kiến trúc, hàm loss, nhãn và pipeline đều
   đúng thì mAP phải tiến gần 1.0.

**Cảnh báo về cách đọc kết quả [3.10]:** đạt phép thử này **không** chứng minh
model phân loại tốt nhà ngập / không ngập. Nó chỉ chứng minh đường ống chạy
đúng. Độ chính xác thật đo ở Phase 5 trên tập val. Tuyệt đối **không** trích dẫn
con số mAP của [3.10] vào báo cáo như một kết quả của đồ án — đó là điểm trên
tập train.


In [ ]:
# [3.8] SINH 20 ẢNH OVERFIT CHO PHÉP THỬ HỌC VẸT
#
# Chọn tất định theo seed: cùng seed thì đúng 20 ảnh đó, mọi lần chạy, trên mọi
# máy. Nhờ vậy nếu phải chạy lại sanity check ở phiên Colab khác thì vẫn là CÙNG
# một phép thử, chứ không phải một phép thử mới.
#
# Tỉ lệ 12 ảnh có nhà ngập / 6 ảnh chỉ có nhà không ngập / 2 ảnh rỗng. Có nhóm
# "ảnh rỗng" là cố ý: đó là nhánh dễ hỏng nhất (mmdet mặc định bỏ ảnh không có
# box nào, mà đồ án cố ý đặt filter_empty_gt=False), nên phép thử phải chạm vào.
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import sys

lenh_overfit = [sys.executable, "-u", "scripts/tao_overfit20.py",
                "--config", DATA_YAML,
                "--processed-dir", PROCESSED_DIR,
                "--split-nguon", "train",
                "--so-anh", "20",
                "--seed", "42"]
print("Chạy: " + " ".join(lenh_overfit))
print("=" * 72)
ma_of = chay_hien_dan(lenh_overfit, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_of, "(0 là thành công)")

if ma_of == 0:
    print("\nTệp annotation sinh ra: "
          f"{PROCESSED_DIR}/annotations/instances_overfit20.json")
    print("Config overfit20.py trỏ thẳng vào tệp này (qua `{{_base_.ann_overfit}}`).")


In [ ]:
# [3.9] ĐO THỜI GIAN 1 EPOCH + VRAM ĐỈNH — số liệu chính của GATE 3
#
# Ô này chạy THẬT vài vòng lặp train (có forward, backward, cập nhật optimizer,
# kể cả AMP và tích luỹ gradient) nhưng KHÔNG ghi checkpoint nào: `scripts/train.py`
# trỏ work_dir vào một thư mục tạm và xoá đi sau khi đo.
#
# Vì sao phải chạy trước khi train thật: một suất train 24 epoch là 6-8 giờ. Biết
# trước 1 epoch mất bao lâu thì mới chốt được số epoch, và biết VRAM còn dư bao
# nhiêu thì mới biết có nguy cơ OOM hay không.
#
# Lần đầu chạy sẽ tải trọng số ImageNet của ConvNeXt-Tiny (~110 MB) nên bước
# dựng model hơi lâu. Đó là chi phí thật, lần train thật cũng trả.
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import sys

lenh_do = [sys.executable, "-u", "scripts/train.py",
           "--config", CFG_CHINH,
           "--data-yaml", DATA_YAML,
           "--dry-run",
           "--so-vong", str(SO_VONG_DO)]
print("Chạy: " + " ".join(lenh_do))
print("=" * 72)
ma_do = chay_hien_dan(lenh_do, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_do, "(0 là tiền kiểm đạt và đo xong)")

if ma_do != 0:
    print("\n" + "!" * 72)
    print("DỪNG LẠI. Tiền kiểm không đạt thì train cũng vô nghĩa.")
    print("Copy nguyên phần lỗi ở trên gửi lại cho Claude.")
    print("!" * 72)
else:
    print("\nGhi lại 3 con số để gửi cho Claude:")
    print("  - thời gian/vòng (ms)")
    print("  - => 1 epoch (phút)")
    print("  - VRAM đỉnh (đã giành chỗ) / tổng VRAM của GPU")


---
## Gửi kết quả cho Claude

Sau khi chạy hết [3.6] → [3.10], copy lại 4 thứ:

1. Toàn bộ output ô **[3.6]** (kiểm tra dữ liệu).
2. Output ô **[3.7]**, nhất là dòng `KẾT LUẬN:` cuối cùng.
3. Output ô **[3.9]**: `thời gian/vòng`, `1 epoch`, `VRAM đỉnh`.
4. Output ô **[3.10]**: bảng mAP theo epoch, và đường dẫn ảnh
   `vis_data` để mở xem model vẽ box có đúng chỗ không.

Đủ 4 thứ đó thì mới chốt được số epoch cho [3.11], và mới biết có phải sửa
`scales_train` hay `batch_size` không.


In [ ]:
# [3.10] SANITY CHECK HỌC VẸT 20 ẢNH (GATE 3)
#
# Train trên 20 ảnh rồi ĐÁNH GIÁ TRÊN CHÍNH 20 ẢNH ĐÓ. Nếu kiến trúc, hàm loss,
# nhãn và pipeline đều đúng thì model phải thuộc lòng được và mAP tiến gần 1.0.
#
# ĐỌC KẾT QUẢ THẾ NÀO:
#   - mAP > 0.9 sau khoảng 60 epoch  -> đường ống đúng. Sang [3.11].
#   - mAP quanh quẩn ở mức thấp      -> có lỗi thật (nhãn sai, box lệch, LR
#                                       hỏng), KHÔNG phải "cần thêm epoch" và
#                                       KHÔNG phải "dữ liệu ít". Dừng lại.
#
# CẢNH BÁO: đạt phép thử này KHÔNG chứng minh model phân loại tốt nhà ngập /
# không ngập — nó chỉ chứng minh đường ống chạy đúng. Tuyệt đối không trích dẫn
# con số mAP ở đây vào báo cáo như một kết quả của đồ án (đó là điểm trên tập
# train, không phải trên tập val).
#
# Chạy hết 60 epoch mất khoảng 20-40 phút. Đây là lần chạy DUY NHẤT có ghi ảnh
# minh hoạ (draw=True) — mở work_dir/vis_data/ để xem model vẽ box có đúng chỗ.
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import os
import sys

if not os.path.exists(f"{PROCESSED_DIR}/annotations/instances_overfit20.json"):
    raise SystemExit("Chưa có instances_overfit20.json — chạy ô [3.8] trước.")

lenh_of = [sys.executable, "-u", "scripts/train.py",
           "--config", CFG_OVERFIT,
           "--data-yaml", DATA_YAML]
print("Chạy: " + " ".join(lenh_of))
print("=" * 72)
ma_of = chay_hien_dan(lenh_of, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_of, "(0 là chạy hết)")
print("\nCheckpoint + ảnh minh hoạ nằm ở:")
print("  " + f"{DRIVE_DIR}/runs/sanity/overfit20")


---
## Bước 3 — Train thật

**Chỉ chạy sau khi cả ba việc của GATE 3 đã đạt.** Cụ thể:

- [3.6] kết thúc bằng `*** ĐẠT HẾT ***`
- [3.7] kết luận dải anchor **PHỦ TỐT**
- [3.9] không có dòng `LOSS = NaN`, VRAM đỉnh còn dư dưới 92%
- [3.10] mAP tiến gần 1.0 ở 20 ảnh overfit

Ô [3.11] **tự resume** nếu `work_dir` trên Drive đã có checkpoint: Colab ngắt
giữa chừng thì cứ chạy lại ô đó, không mất kết quả. Checkpoint ghi thẳng ra Drive
vì `/content` mất sạch khi hết phiên.

Thời gian ước tính lấy từ chính ô [3.9], không đoán.


In [ ]:
# [3.11] TRAIN THẬT — CHỈ CHẠY SAU KHI GATE 3 ĐẠT
#
# Kiểm lại lần cuối trước khi đốt 6-8 giờ GPU:
#   [3.6]  kết thúc bằng *** ĐẠT HẾT ***
#   [3.7]  kết luận dải anchor PHỦ TỐT
#   [3.9]  không có dòng LOSS = NaN, VRAM đỉnh còn dư dưới 92%
#   [3.10] mAP tiến gần 1.0 trên 20 ảnh overfit
#   Và SO_EPOCH ở ô [3.1] đã được chốt theo số đo của [3.9].
#
# TỰ RESUME: nếu work_dir trên Drive đã có checkpoint, runner tự động chạy tiếp
# từ bản mới nhất. Colab ngắt giữa chừng thì cứ chạy lại ô này, không mất kết
# quả. Muốn train lại từ đầu thì xoá thư mục work_dir trên Drive.
#
# KHÔNG dừng Colab bằng cách đóng tab — checkpoint ghi mỗi epoch nên mất tối đa
# một epoch. Nhưng nhớ rằng /content mất sạch: phiên sau phải chạy lại [3.2]-[3.5].
if "chay_hien_dan" not in globals():
    raise SystemExit("Chưa chạy ô [3.1]-[3.4].")

import sys

lenh_train = [sys.executable, "-u", "scripts/train.py",
              "--config", CFG_CHINH,
              "--data-yaml", DATA_YAML,
              "--max-epochs", str(SO_EPOCH),
              # Đối chiếu TỪNG ảnh lúc tiền kiểm (chậm hơn vài chục giây, nhưng
              # đây là lần chạy đắt tiền nhất nên cứ kiểm kỹ).
              "--kiem-cham"]
print("Chạy: " + " ".join(lenh_train))
print("=" * 72)
ma_train = chay_hien_dan(lenh_train, REPO_DIR)
print("=" * 72)
print("Mã thoát:", ma_train)
print("\nCheckpoint nằm ở thư mục work_dir trong config:")
print("  " + f"{DRIVE_DIR}/runs/train/e1_cascade_convnext_t")
